# Demographic Robustness Baseline (Section 4.X)

**목적**: 비교군 4개 ECG foundation 모델 (CPC, ECG-FM, ECG-Founder, ECG-JEPA)이
진단을 demographic (성인/소아)에 robust하게 표현하는지 UMAP + 정량 지표로 평가.
추후 우리 모델이 더 robust함을 입증하기 위한 baseline.

## 평가 setup

- **모델**: CPC, ECG-FM, ECG-Founder, ECG-JEPA
- **데이터셋**: PTB-XL (성인 21,410 + 소아 426) + ZZU-pECG (모두 소아 12,327)
- **라벨 통일**: ICD-10 prefix 매핑
  - PTBXL paper labels (약자) → ICD prefix
  - ZZU bench labels (`is_*`) → ICD prefix (사용자 매핑)
- **분석**: 각 진단 코드별 `silhouette` + `kNN-BACC` (one-vs-rest)
- **subgroup**: adult ≥18 / pediatric <18 / combined

## 핵심 메시지 (검증할 것)

1. UMAP에서 같은 진단 코드를 가진 샘플은 demographic 차이를 넘어 함께 클러스터링되는가?
2. **subgroup gap** (`|adult_BACC − pediatric_BACC|`)이 작은가?
3. **worst-group BACC** (= min(adult, pediatric))가 높은가?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from scripts.umap_view import (
    quick_dx, get_dx_codes, ICD_DISPLAY, list_models,
    dx_compatibility_table,
)
from collections import Counter
import pandas as pd
import numpy as np
pd.set_option('display.width', 140)
pd.set_option('display.max_rows', 50)

print('models:', list_models())
print('datasets: ptbxl, zzu')

## 1. 진단 카테고리 — 두 데이터셋 호환성 표

ICD-10 prefix별 PTB-XL / ZZU 등장 횟수.
**`compatible=True`** 행만 cross-dataset 비교가 의미 있음 (둘 다 sample 존재).

- 호환 (12개): `Normal/SR, S.Tachy, S.Brady, S.Arr, 1° AVB, LAFB, RBBB, WPW, Long QT, SVT, Hypertrophy/Enlarge, Other ab. ECG`
- PTBXL-only (7): `AFib, LPFB, LBBB, Other IVCD, PAC, PVC, Paced`
- ZZU-only (6): `Complete AVB, AV Dissoc, Junct/AEsc, Hyper/Hypo-K/Ca`

In [ ]:
compat = dx_compatibility_table()
print(compat.to_string(index=False))
print(f"\n호환 {compat.compatible.sum()} / 전체 {len(compat)}")

In [ ]:
# === Setup: 이 한 셀로 아래 quick_dx 호출에 필요한 모든 것 준비 ===
import sys, os, datetime
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import scripts.umap_view as uv
from scripts.umap_view import quick_dx, dx_compatibility_table, ICD_DISPLAY

# 논문용 rcParams — TrueType 폰트 임베딩 (Illustrator 편집 가능, 깨짐 X)
plt.rcParams.update({
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.03,
    'figure.autolayout': False,
})

PROJECT_ROOT = Path(os.path.abspath('..'))
RESULTS_ROOT = PROJECT_ROOT / 'results'

# 1) 호환 진단 코드
compat = dx_compatibility_table()
compat_codes = compat[compat['compatible']]['icd'].tolist()

# 2) 모델 행 라벨 표시 이름
DISPLAY_NAMES = {
    'CPC':         'ECG-CPC',
    'Ours-cb1024': 'MoRyECG(Ours)',
}

# 3) 논문용 wrapper — 제목 X, metrics 표 X, legend 별도 figure
#    layout='2x6': 가로 긴 2줄 × (모델 2개 × age 3) = 6 col / 2 row  (default)
#    layout='4x3': 세로 긴 4 모델 행 × 3 age col
#    저장: results/YYYYMMDD_HHMMSS/{name}.{pdf,svg,png}  — pdf/svg = vector (무손실 zoom)
def quick_dx_pretty(*args,
                    layout='2x6',
                    figsize_per_cell=(1.9, 1.9),
                    dpi=110,
                    label_fontsize=11,
                    legend_fontsize=10,
                    save_name='dx_umap',
                    save_dir=None,
                    **kwargs):
    kwargs['show'] = False
    kwargs['show_metrics'] = False
    kwargs.setdefault('figsize_per_cell', figsize_per_cell)

    captured = {'handles': []}
    def _capture_legend(self, *a, **kw):
        h = kw.get('handles') or (a[0] if a else None) or []
        captured['handles'] = list(h)
        return None
    _orig_legend, _orig_suptitle = Figure.legend, Figure.suptitle
    Figure.legend = _capture_legend
    Figure.suptitle = lambda self, *a, **kw: None
    try:
        fig, metrics = quick_dx(*args, **kwargs)
    finally:
        Figure.legend, Figure.suptitle = _orig_legend, _orig_suptitle

    fig.set_dpi(dpi)
    n_age = 3
    n_models = len(fig.axes) // n_age
    raw_axes = list(fig.axes)
    titles_per_age = [
        (raw_axes[a].get_title()
            .replace('adult ≥18.0', 'adult').replace('adult ≥18', 'adult')
            .replace('pediatric <18.0', 'pediatric').replace('pediatric <18', 'pediatric'))
        for a in range(n_age)
    ]
    for ax in raw_axes:
        cur = ax.get_ylabel()
        if cur in DISPLAY_NAMES:
            ax.set_ylabel(DISPLAY_NAMES[cur], fontweight='bold',
                          fontsize=label_fontsize, rotation=0, labelpad=8,
                          ha='right', va='center')

    if layout == '2x6':
        models_per_row = 2
        tgt_rows = (n_models + models_per_row - 1) // models_per_row
        tgt_cols = models_per_row * n_age
        fig.set_size_inches(figsize_per_cell[0] * tgt_cols + 0.6,
                            figsize_per_cell[1] * tgt_rows + 0.5)
        left, right, top, bottom = 0.04, 0.995, 0.90, 0.05
        wspace, hspace = 0.04, 0.22
        cw = (right - left - wspace * (tgt_cols - 1)) / tgt_cols
        ch = (top - bottom - hspace * (tgt_rows - 1)) / tgt_rows
        for i, ax in enumerate(raw_axes):
            m, a = i // n_age, i % n_age
            nr = m // models_per_row
            nc = (m % models_per_row) * n_age + a
            l = left + nc * (cw + wspace)
            b = top - (nr + 1) * ch - nr * hspace
            ax.set_position([l, b, cw, ch])
            ax.set_title(titles_per_age[a], fontweight='bold',
                         fontsize=label_fontsize, pad=4)
    else:  # '4x3'
        gs = raw_axes[0].get_subplotspec().get_gridspec()
        gs.update(left=0.10, right=0.99, top=0.95, bottom=0.04,
                  hspace=0.10, wspace=0.05)
        fig.set_size_inches(figsize_per_cell[0] * 3 + 0.9,
                            figsize_per_cell[1] * gs.nrows + 0.5)
    plt.show()

    # 별도 legend figure (vector PDF 동일 화질)
    handles = captured['handles']
    labels = [h.get_label() for h in handles]
    n = len(handles)
    leg_fig = plt.figure(figsize=(2.4, max(1.5, 0.24 * n + 0.3)), dpi=dpi)
    leg_fig.legend(handles=handles, labels=labels,
                   loc='center', ncol=1,
                   fontsize=legend_fontsize, frameon=False,
                   handlelength=1.0, handletextpad=0.5, labelspacing=0.7)
    plt.show()

    # 저장: pdf/svg (vector, 무한 zoom) + png 600dpi (raster fallback)
    if save_dir is None:
        save_dir = RESULTS_ROOT / datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    for ext, save_dpi in [('pdf', None), ('svg', None), ('png', 600)]:
        kw = {'dpi': save_dpi} if save_dpi else {}
        fig.savefig(save_dir / f'{save_name}.{ext}', **kw)
        leg_fig.savefig(save_dir / f'{save_name}_legend.{ext}', **kw)
    print(f'[saved] {save_dir}/')
    print(f'  {save_name}.{{pdf,svg,png}}  +  {save_name}_legend.{{pdf,svg,png}}')
    print(f'  논문 용도: pdf 또는 svg (vector — 축소해도 화질 유지)')

    return fig, leg_fig, {DISPLAY_NAMES.get(k, k): v for k, v in metrics.items()}, save_dir

print('compat_codes:', compat_codes)
print('models for next cell: CPC→ECG-CPC, ECG-FM, ECG-JEPA, Ours-cb1024→MoRyECG(Ours)')
print(f'output base: {RESULTS_ROOT}/<YYYYMMDD_HHMMSS>/')

## 2. UMAP — 4 모델 × (combined / adult / pediatric)

**호환 진단 코드만** 색칠. `balance_per_code` 로 진단별 최대 샘플 수를 균등화하여
Normal 비대화로 인한 가독성 저하를 방지.

각 셀에서 해당 그룹만 진단별 색칠 (balance 적용), 다른 그룹/미매핑은 회색.
오른쪽엔 combined 그룹의 코드별 `silhouette` / `kNN-BACC`.

> **balance_per_code=300** = 각 진단당 최대 300개만 표시.
> 작은 진단(WPW 6 등)은 그대로, 큰 진단(Normal 18,058)은 300개로 줄임.

In [ ]:
# 호환 12개 진단 — 진단당 max 500 샘플로 balance
fig, leg_fig, metrics, save_dir = quick_dx_pretty(
    models=['CPC', 'ECG-FM', 'ECG-JEPA', 'Ours-cb1024'],
    include_codes=compat_codes,
    age_split=18.0,
    balance_per_code=500,
    save_name='dx_umap_compat12',
)

In [ ]:
# 임상적으로 핵심인 부정맥/전도장애 (호환 코드 중) — balance 적용
fig2, metrics_focused = quick_dx(
    include_codes=['I45.1', 'I44.0', 'I44.4', 'I45.6', 'I45.81', 'I47.1'],
    age_split=18.0,
    balance_per_code=300,
)

### 2-1) Normal 제외 — 비정상 진단만 색칠

Normal/SR 이 너무 많아서 다른 진단이 묻히는 경우, `exclude_codes=['Z00.00']` 로
Normal 을 회색 배경에 포함시키고 비정상 진단들만 강조한다.

In [ ]:
# Normal/SR 빼고 나머지 진단만 색칠 (Normal 은 회색)
fig, metrics_no_normal = quick_dx(
    include_codes=compat_codes,
    exclude_codes=['Z00.00'],
    age_split=18.0,
    balance_per_code=500,
)

## 3. Subgroup gap 분석 — adult vs pediatric BACC

각 모델·진단별 `adult_BACC`, `pediatric_BACC`, `gap`, `worst-group BACC`.

- **gap이 작을수록** demographic-robust
- **worst가 높을수록** minority subgroup (소아) 성능 보장
- 두 지표를 합친 모델이 representation-level robust 하다는 evidence

In [ ]:
def subgroup_table(metrics_dict):
    """quick_dx 가 반환한 metrics → adult/pediatric BACC, gap, worst 표."""
    adult_key = next(k for k in next(iter(metrics_dict.values())) if 'adult' in k)
    ped_key   = next(k for k in next(iter(metrics_dict.values())) if 'pediatric' in k)
    rows = []
    for model, by_grp in metrics_dict.items():
        for code, mm_a in by_grp.get(adult_key, {}).items():
            mm_p = by_grp.get(ped_key, {}).get(code, {})
            ba, bp = mm_a.get('bacc'), mm_p.get('bacc')
            if ba is None or bp is None or not (np.isfinite(ba) and np.isfinite(bp)):
                continue
            rows.append({
                'model': model, 'code': code,
                'dx': ICD_DISPLAY.get(code, code),
                'adult_bacc': round(ba, 3),
                'pedi_bacc':  round(bp, 3),
                'gap':        round(abs(ba - bp), 3),
                'worst':      round(min(ba, bp), 3),
                'n_adult':    int(mm_a.get('n_pos', 0)),
                'n_pedi':     int(mm_p.get('n_pos', 0)),
            })
    return pd.DataFrame(rows)

In [ ]:
df = subgroup_table(metrics)
summary = df.groupby('model')[['gap', 'worst']].mean().round(3)
summary = summary.reindex(['CPC', 'ECG-FM', 'ECG-Founder', 'ECG-JEPA'])
summary['rank_gap']   = summary['gap'].rank()        # 작을수록 1위
summary['rank_worst'] = summary['worst'].rank(ascending=False)
print('=== 모델별 평균 (낮은 gap + 높은 worst가 좋음) ===')
print(summary)

In [ ]:
# 진단별 상세 (모델 × 진단)
df_pivot_gap = df.pivot_table(index='dx', columns='model', values='gap')
df_pivot_worst = df.pivot_table(index='dx', columns='model', values='worst')
print('=== gap (낮을수록 좋음) ===')
print(df_pivot_gap.round(3))
print('\n=== worst-group BACC (높을수록 좋음) ===')
print(df_pivot_worst.round(3))

## 4. 부정맥 focused subgroup 분석

Section 2의 두 번째 figure (호환 진단 중 핵심 부정맥)에 대한 subgroup 표.

In [ ]:
df_focused = subgroup_table(metrics_focused)
print('=== 부정맥 focused: 모델 × 진단 gap ===')
print(df_focused.pivot_table(index='dx', columns='model', values='gap').round(3))
print('\n=== worst-group BACC ===')
print(df_focused.pivot_table(index='dx', columns='model', values='worst').round(3))

## 6) 보완 시각화

- **6-1**: paper label 약자 그대로 (ICD 통합 없이) — 단일 데이터셋 raw 진단
- **6-2**: (성인 × 진단) / (소아 × 진단) **셀별 sample 수 균등화** — fair subgroup 비교

In [ ]:
# 6-1) PTBXL paper label 약자 그대로 (한 줄 가로 legend)
from scripts.umap_view import quick_dx_raw

quick_dx_raw(
    'ECG-Founder', 'ptbxl',
    label_columns=['AFIB','SR','CRBBB','CLBBB','PVC','PAC','STACH'],
    max_per_label=300,
)

In [ ]:
# 6-1') chapman paper label — 이미지 형식 그대로 (한 줄 가로 legend)
# SA 는 chapman_paper_labels.csv 에 없어 SR(Sinus rhythm)으로 대체
quick_dx_raw(
    'ECG-Founder', 'chapman',
    label_columns=['SR', 'ALS', 'APB', 'AF', 'SVT', 'TWC', 'ST'],
    max_per_label=300,
)

In [ ]:
# 6-1'') 사용자 약자 → 의미 → PTBXL/chapman 컬럼 → 우리 ICD 매핑 표
from scripts.umap_view import ICD_LABEL_MAP, ICD_DISPLAY
import pandas as pd

user_codes = [
    # (약자, 의미, chapman 컬럼, PTBXL 컬럼, ICD prefix)
    ('SA',  'Sinus arrhythmia',          None,  'SARRH', 'R00.8'),
    ('ALS', 'chapman 고유 (T-axis 변이)',  'ALS',  None,    None),
    ('APB', 'Atrial Premature Beat',     'APB',  'PAC',   'I49.1'),
    ('AF',  'Atrial Fibrillation',       'AF',   'AFIB',  'I48'),
    ('SVT', 'Supraventricular Tachy',    'SVT',  'PSVT',  'I47.1'),
    ('TWC', 'T Wave Change',             'TWC',  None,    None),
    ('ST',  'Sinus Tachycardia',         'ST',   'STACH', 'R00.0'),
]
ptbxl_map = ICD_LABEL_MAP['ptbxl']
rows = []
for short, meaning, chap, ptb, icd in user_codes:
    rows.append({
        'short': short,
        'meaning': meaning,
        'chapman': chap or '-',
        'ptbxl':   ptb  or '-',
        'ICD':     icd  or '-',
        'mapped_to_ICD_LABEL_MAP': bool(ptb and ptbxl_map.get(ptb)),
        'display':  ICD_DISPLAY.get(icd, '-') if icd else '-',
    })
pd.DataFrame(rows).to_string(index=False)
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))

In [ ]:
# 6-2) (성인/소아 × 진단)별 sample 수 균등 — 각 셀당 max 100개
# adult AFib 100 / pediatric AFib 100 / adult RBBB 100 / ...
fig, metrics_grpcap = quick_dx(
    include_codes=compat_codes,
    exclude_codes=['Z00.00'],   # Normal/SR 회색
    age_split=18.0,
    balance_per_group_code=100,
)

In [ ]:
# 6-3) strict 균등화 — adult vs pediatric 진단별 sample 수를 자동으로 작은 쪽에 통일
# 명시적 cap 숫자 없이, 각 진단마다 min(adult, ped) 만큼만 양쪽 표시.
fig, metrics_strict = quick_dx(
    include_codes=compat_codes,
    exclude_codes=['Z00.00'],
    age_split=18.0,
    balance_strict_groups=True,
)

## 7) PTBXL super class (NORM/MI/STTC/CD/HYP) — ZZU 도 같은 5종으로 묶기

PTBXL super 5종은 ECG 진단의 가장 통상적인 4.X 분석 단위.
ZZU bench labels (`is_*`) 를 super class에 매핑해 두 데이터셋을 같은 5색으로 시각화.

**ZZU → super 매핑** (`scripts/umap_view.py` `SUPER_LABEL_MAP`):
- NORM: `is_Normal | is_Normal_Other`
- STTC: `is_TAb, is_ST_T, is_ST, is_STDep, is_STElev, is_QAb, is_UWave, is_EarlyRepol`
- CD: `is_RBBB, is_IRBBB, is_LAnFB, is_LQT/c, is_IAVB, is_CAVB, is_WPW, is_AVDiss, is_JTach, is_JEsc, is_EATach, is_AEsc`
- HYP: `is_RVH, is_LVH, is_LVHV, is_RAE, is_LAE`
- MI: ZZU에 직접 라벨 없음 (소아 데이터셋 특성)

In [ ]:
from scripts.umap_view import get_super_codes, SUPER_DISPLAY
from collections import Counter

for d, n in [('ptbxl', 21836), ('zzu', 12327)]:
    codes, _ = get_super_codes(d, n)
    cnt = Counter(c for c in codes if c)
    print(f'\n[{d}] super coded={n - codes.count(None):,}/{n:,}')
    for code, c in cnt.most_common():
        print(f'  {code:6} {SUPER_DISPLAY.get(code, "?"):25} {c:6,}')

In [ ]:
quick_dx(code_scheme='super',
         include_codes=['NORM','STTC','CD','HYP', 'MI'],
         age_split=18.0, balance_strict_groups=True)

In [ ]:
quick_dx(code_scheme='super',
         include_codes=['NORM','STTC','CD','HYP'],
         age_split=18.0, balance_strict_groups=True)

In [ ]:
# 4 모델 × (combined/adult/pediatric) — super 5종, 양쪽 그룹 sample 수 자동 균등
fig, metrics_super = quick_dx(
    datasets=('ptbxl', 'zzu'),
    code_scheme='super',
    include_codes=['NORM', 'STTC', 'CD', 'HYP', 'MI'],
    age_split=18.0,
    balance_strict_groups=True,
)

In [ ]:
# 모델·super 진단별 adult vs pediatric BACC
import pandas as pd
import numpy as np
rows = []
for model, by_grp in metrics_super.items():
    a, p = by_grp.get('adult ≥18', {}), by_grp.get('pediatric <18', {})
    for code in ['NORM', 'STTC', 'CD', 'HYP', 'MI']:
        ba = a.get(code, {}).get('bacc')
        bp = p.get(code, {}).get('bacc')
        if ba is None or bp is None: continue
        rows.append({
            'model': model, 'super': code,
            'name': SUPER_DISPLAY.get(code),
            'adult': round(ba, 3) if np.isfinite(ba) else None,
            'pedi':  round(bp, 3) if np.isfinite(bp) else None,
            'gap':   round(abs(ba-bp), 3) if (np.isfinite(ba) and np.isfinite(bp)) else None,
            'worst': round(min(ba, bp), 3) if (np.isfinite(ba) and np.isfinite(bp)) else None,
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print('\n=== 모델별 평균 (gap↓ / worst↑) ===')
print(df.groupby('model')[['gap','worst']].mean().round(3))

## 5. 결론 (보고서용 메모)

위 결과의 해석을 4.X 섹션 작성에 사용:

- **"adult-pediatric gap이 가장 작은 모델"** = `summary['gap']` 최솟값 모델
- **"worst-group BACC가 가장 높은 모델"** = `summary['worst']` 최댓값 모델
- 두 지표 모두 1위인 모델이 representation-level demographic-robust

**호환 진단 12개**:
- 정상: Normal/SR (Z00.00)
- 동성 리듬 변이: S.Tachy (R00.0), S.Brady (R00.1), S.Arr (R00.8)
- 전도 장애: 1° AVB (I44.0), LAFB (I44.4), RBBB (I45.1)
- 빈맥: SVT (I47.1)
- preexcitation: WPW (I45.6)
- 재분극: Long QT (I45.81)
- 비대/확장: Hypertrophy/Enlarge (I51.7)
- 비특이: Other ab. ECG (R94.31)

추후 우리 모델 추가 시:
```python
from scripts.umap_view import list_models
list_models(exclude=[])  # 모든 모델 포함
# 또는: quick_dx(models=['CPC','ECG-FM','ECG-Founder','ECG-JEPA','MyModel'], ...)
```